# Geophysical Waveform Inversion: Kaggle Pipeline

Run the cells from top to bottom. The notebook downloads the GitHub repository when needed, validates the environment and data, computes statistics, performs a one-epoch preflight, and provides separate cells for formal training, inference, and submission validation.

In [ ]:
# Kaggle setup: use the current repository or clone it from GitHub.
from pathlib import Path
import os
import sys
import subprocess

REPO_URL = "https://github.com/fancyleo/Geophysical-Waveform-Inversion.git"
CURRENT_DIR = Path.cwd()
REPO_DIR = next((path for path in [CURRENT_DIR, *CURRENT_DIR.parents] if (path / "working_space" / "train.py").exists()), CURRENT_DIR / "Geophysical-Waveform-Inversion")
if not (REPO_DIR / "working_space" / "train.py").exists():
    subprocess.run(["git", "clone", "--depth", "1", REPO_URL, str(REPO_DIR)], check=True)

# Kaggle mounts competition data separately from the code repository.
KAGGLE_DATA_ROOT = Path("/kaggle/input/competitions/waveform-inversion")
if not (KAGGLE_DATA_ROOT / "train_samples").is_dir():
    raise FileNotFoundError(f"Training data not found: {KAGGLE_DATA_ROOT / 'train_samples'}")
if not (KAGGLE_DATA_ROOT / "test").is_dir():
    raise FileNotFoundError(f"Test data not found: {KAGGLE_DATA_ROOT / 'test'}")

os.environ["WAVEFORM_DATA_ROOT"] = str(KAGGLE_DATA_ROOT)
os.environ["WAVEFORM_OUTPUT_ROOT"] = "/kaggle/working"
# Reduce CUDA allocator fragmentation during large activation allocations.
os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "expandable_segments:True")
# Disable tqdm bars in training subprocesses to avoid flooding the cell output buffer.
os.environ["TQDM_DISABLE"] = "1"
# Limit glibc malloc arenas so freed memory is returned to the OS instead of
# accumulating in process RSS across epochs.
os.environ.setdefault("MALLOC_ARENA_MAX", "2")
sys.path.insert(0, str(REPO_DIR / "working_space"))
print(f"Repository: {REPO_DIR}")
print(f"Input root: {os.environ['WAVEFORM_DATA_ROOT']}")
print(f"Output root: {os.environ['WAVEFORM_OUTPUT_ROOT']}")
print(f"CUDA allocator: {os.environ['PYTORCH_CUDA_ALLOC_CONF']}")
print(f"tqdm disabled: {os.environ['TQDM_DISABLE']}")
print(f"MALLOC_ARENA_MAX: {os.environ['MALLOC_ARENA_MAX']}")

In [ ]:
# Copy the repository's working_space contents into /kaggle/working.
import shutil

WORKING_SPACE_DIR = REPO_DIR / "working_space"
KAGGLE_WORKING_DIR = Path("/kaggle/working")
KAGGLE_WORKING_DIR.mkdir(parents=True, exist_ok=True)

shutil.copytree(
    WORKING_SPACE_DIR,
    KAGGLE_WORKING_DIR,
    dirs_exist_ok=True,
)
print(f"Copied: {WORKING_SPACE_DIR}")
print(f"Destination: {KAGGLE_WORKING_DIR}")

In [ ]:
import importlib
import torch

for package_name in ["numpy", "matplotlib", "sklearn", "tqdm", "torch"]:
    module = importlib.import_module(package_name)
    print(f"{package_name}: {getattr(module, '__version__', 'available')}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

In [ ]:
from config import Cfg, select_families
from train import find_pairs
import os

print(f"Input root: {Cfg.input_root}")
print(f"Training data: {Cfg.train_data_dir}")
print(f"Test data: {Cfg.test_data_dir}")
print(f"Training data exists: {Cfg.train_data_dir.is_dir()}")
print(f"Test data exists: {Cfg.test_data_dir.is_dir()}")

# Diagnostic: show what is actually under the training data root.
train_root = str(Cfg.train_data_dir)
if os.path.isdir(train_root):
    entries = sorted(os.listdir(train_root))
    print(f"[diag] train_samples entries ({len(entries)}): {entries[:20]}")
    for entry in entries[:3]:
        sub = os.path.join(train_root, entry)
        if os.path.isdir(sub):
            print(f"[diag]   {entry}/ -> {sorted(os.listdir(sub))[:10]}")

families = select_families("all")
pairs = find_pairs(train_root, families)
print(f"Families: {len(families)}; paired files: {len(pairs)}")
assert pairs, "No training pairs were found."


In [ ]:
subprocess.run([sys.executable, str(REPO_DIR / "working_space" / "test_unet.py")], check=True)
subprocess.run([sys.executable, str(REPO_DIR / "working_space" / "smoke_test.py")], check=True)

In [ ]:
# Ensure the velocity statistics JSON exists before training so train.py loads
# the true dataset mean/std instead of falling back to Cfg.vel_mean / vel_std
# (which would also emit the "[warn] statistics file not found" message).
from config import Cfg, select_families, stats_path_for_families, load_velocity_stats

families = select_families("all")
stats_path = stats_path_for_families(families)
if stats_path.is_file():
    print(f"[info] using existing velocity statistics: {stats_path}")
else:
    print(f"[info] computing velocity statistics: {stats_path}")
    subprocess.run(
        [sys.executable, str(REPO_DIR / "working_space" / "compute_stats.py"),
         "--data_dir", str(Cfg.train_data_dir), "--family", "all"],
        check=True,
    )
vel_mean, vel_std = load_velocity_stats(stats_path)
print(f"[info] velocity mean={vel_mean:.2f}  std={vel_std:.2f}")
assert stats_path.is_file(), f"Statistics still missing: {stats_path}"

In [ ]:
# Run a multi-GPU DDP smoke test before the formal training job.
# train.py --test_run uses the flat families, 3 epochs, and memory monitoring;
# --parallel_mode data_parallel with --nproc_per_node 2 launches DDP over the
# available T4 GPUs. Watch the [mem] delta: it should stay flat across epochs,
# otherwise a host memory leak is present and you should fall back to single.
TRAIN_DIR = Cfg.train_data_dir
OUTPUT_DIR = Cfg.output_dir
NPROC = 2 if torch.cuda.device_count() > 1 else 1
subprocess.run([
    sys.executable, str(REPO_DIR / "working_space" / "train.py"),
    "--data_dir", str(TRAIN_DIR), "--out_dir", str(OUTPUT_DIR),
    "--test_run",
    "--batch_size", "4",
    "--num_workers", "0",
    "--parallel_mode", "data_parallel",
    "--nproc_per_node", str(NPROC),
    "--log_memory",
], check=True)
print(f"[info] DDP smoke test finished with nproc={NPROC}")

In [ ]:
# Run this cell for the formal training job.
# single mode is stable on host memory (verified locally). If the DDP smoke
# test above showed a flat [mem] delta across epochs, you can enable parallel
# training by using --parallel_mode data_parallel --nproc_per_node 2.
EPOCHS = 30
BATCH_SIZE = 4
FORMAL_PARALLEL_MODE = "single"   # switch to "data_parallel" after a clean DDP smoke test
FORMAL_NPROC = 2 if FORMAL_PARALLEL_MODE in ("data_parallel", "ddp") and torch.cuda.device_count() > 1 else 1
subprocess.run([
    sys.executable, str(REPO_DIR / "working_space" / "train.py"),
    "--data_dir", str(TRAIN_DIR), "--out_dir", str(OUTPUT_DIR),
    "--family", "all", "--epochs", str(EPOCHS),
    "--batch_size", str(BATCH_SIZE),
    "--num_workers", "0",
    "--parallel_mode", FORMAL_PARALLEL_MODE,
    "--nproc_per_node", str(FORMAL_NPROC),
    "--log_memory",
], check=True)

In [ ]:
run_dirs = sorted(OUTPUT_DIR.glob("**/model_*/"), key=lambda path: path.stat().st_mtime)
assert run_dirs, "No training run directory was found."
latest_run = run_dirs[-1]
checkpoint = latest_run / "best_unet.pth"
assert checkpoint.exists(), f"Checkpoint not found: {checkpoint}"
submission_path = OUTPUT_DIR / "submission.csv"
print(f"Using checkpoint: {checkpoint}")
subprocess.run([
    sys.executable, str(REPO_DIR / "working_space" / "infer.py"),
    "--ckpt", str(checkpoint), "--test_dir", str(Cfg.test_data_dir),
    "--out", str(submission_path), "--batch_size", str(Cfg.infer_batch_size),
], check=True)

In [ ]:
import pandas as pd
submission = pd.read_csv(submission_path)
expected_columns = 1 + len(range(Cfg.submission_x_start, Cfg.submission_x_stop, Cfg.submission_x_step))
print(f"Submission shape: {submission.shape}")
print(f"Missing values: {int(submission.isna().sum().sum())}")
assert submission.shape[1] == expected_columns
assert submission.isna().sum().sum() == 0
print("Submission validation passed.")